In [1]:
from pathlib import Path
import sys,csv,time,warnings,numpy as np
from PIL import Image
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score
warnings.filterwarnings("ignore")
R=Path.cwd()
# Notebooku sci5-wbc disinda actiysan buraya proje klasorunun tam yolunu yaz.
PROJECT_ROOT_OVERRIDE = ""
if PROJECT_ROOT_OVERRIDE.strip():
 R=Path(PROJECT_ROOT_OVERRIDE)
if not (R/"data").exists() or not (R/"RETFound").exists():
 raise FileNotFoundError(f"Proje kok klasoru yanlis: {R}. Notebooku sci5-wbc icinde acin veya PROJECT_ROOT_OVERRIDE yazin.")
O=R/"outputs/asama15c_external"
if not O.exists():
 raise FileNotFoundError(f"Eski external test ciktisi yok: {O}")
O_NEW=R/"review_results_external_BNE_SNE_full13"
O_NEW.mkdir(parents=True, exist_ok=True)
CK=R/"wbc_project/checkpoints";sys.path.insert(0,str(R/"RETFound"));import models_vit
DEV="cuda" if torch.cuda.is_available() else "cpu";print("DEVICE",DEV)
SEEDS=[42,1,7]; FULL=['BA','BL','BNE','EO','LY','MMY','MO','MY','PC','PLY','PMY','SNE','VLY']
KEEP=['BA','EO','LY','MO','SNE']; IDX=[FULL.index(x) for x in KEEP]
NAMES=['basophil','eosinophil','lymphocyte','monocyte','neutrophil']
def ckpt(m,s):
 # Explicit locked-checkpoint resolution. Seed42 is in checkpoints root;
 # seeds 1/7 may be stored in final-training output folders rather than that root.
 import re
 aliases={'convnext_tiny':['convnext'],'resnet50':['resnet'],'retfound_fullft':['retfound']}
 search_roots=[R/"wbc_project/checkpoints", R/"outputs", R/"wbc_project"]
 seen=set(); candidates=[]
 for root in search_roots:
  if not root.exists(): continue
  for p in root.rglob("*"):
   if not (p.is_file() and p.suffix.lower() in {'.pth','.pt','.ckpt'}): continue
   rp=str(p.resolve())
   if rp in seen: continue
   seen.add(rp)
   low=rp.lower()
   if not any(a in low for a in aliases[m]): continue
   patterns=[
    rf'(?<![a-z0-9])seed[_\- ]*{s}(?!\d)',
    rf'(?<![a-z0-9])s[_\- ]*{s}(?!\d)',
    rf'(?<!\d){s}[_\- ]*seed(?!\d)'
   ]
   if any(re.search(q,low) for q in patterns):
    candidates.append(p)
 if not candidates:
  avail=[]
  for root in search_roots:
   if root.exists():
    avail += [str(p) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {'.pth','.pt','.ckpt'} and any(a in str(p).lower() for a in aliases[m])]
  print("\\nAVAILABLE FILES FOR",m)
  for x in avail: print(x)
  raise FileNotFoundError(f"{m} seed{s} not found in checkpoints/outputs/wbc_project. See listing above.")
 # Prefer explicit best checkpoint, then final-related path, then shortest path.
 candidates.sort(key=lambda p:(0 if 'best' in p.name.lower() else 1,
                               0 if 'final' in str(p).lower() else 1,
                               len(str(p))))
 print("resolved:",m,"seed",s,"->",candidates[0])
 return candidates[0]
def sd(p):
 x=torch.load(p,map_location="cpu")
 if isinstance(x,dict):
  for k in ['model','state_dict','model_state_dict']:
   if k in x and isinstance(x[k],dict):x=x[k];break
 return {k.removeprefix("module."):v for k,v in x.items()}
def model(n,p):
 if n=='convnext_tiny':
  m=models.convnext_tiny(weights=None);m.classifier[2]=nn.Linear(m.classifier[2].in_features,13)
 elif n=='resnet50':
  m=models.resnet50(weights=None);m.fc=nn.Linear(m.fc.in_features,13)
 else:m=models_vit.RETFound_mae(img_size=224,num_classes=13,drop_path_rate=0)
 a,b=m.load_state_dict(sd(p),strict=False);print("checkpoint",p.name,"missing",len(a),"unexpected",len(b))
 return m.to(DEV).eval()
tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
PBC=R/"data/external/PBC_dataset_normal_DIB";RAA=R/"data/external/Raabin_WBC"
def pbc():
 return [(p,y) for y,n in enumerate(NAMES) for p in sorted((PBC/n).rglob("*")) if p.is_file() and p.suffix.lower() in EXT and "__MACOSX" not in p.parts]
def raabin():
 out=[];A={n.lower():i for i,n in enumerate(NAMES)}
 for split in ['Train','TestA']:
  base=RAA/split
  for p in sorted(base.rglob("*")):
   if p.is_file() and p.suffix.lower() in EXT:
    h=[A[x.lower()] for x in p.relative_to(base).parts[:-1] if x.lower() in A]
    if h:out.append((p,h[-1]))
 return out
class DS(Dataset):
 def __init__(self,z):
  self.z=[];self.bad=[]
  for p,y in z:
   try:
    with Image.open(p) as im:im.verify()
    self.z.append((p,y))
   except Exception as e:self.bad.append((str(p),repr(e)))
 def __len__(self):return len(self.z)
 def __getitem__(self,i):
  p,y=self.z[i]
  with Image.open(p) as im:x=tf(im.convert("RGB"))
  return x,y,str(p)
D={'PBC':DS(pbc()),'Raabin_Train_TestA':DS(raabin())}
print("=== EXTERNAL MANIFEST ===")
for n,d in D.items():print(n,len(d),"bad",len(d.bad),dict(zip(NAMES,np.bincount([y for _,y in d.z],minlength=5))))


DEVICE cuda
=== EXTERNAL MANIFEST ===
PBC 10298 bad 1 {'basophil': np.int64(1218), 'eosinophil': np.int64(3117), 'lymphocyte': np.int64(1214), 'monocyte': np.int64(1420), 'neutrophil': np.int64(3329)}
Raabin_Train_TestA 14514 bad 0 {'basophil': np.int64(301), 'eosinophil': np.int64(1066), 'lymphocyte': np.int64(3461), 'monocyte': np.int64(795), 'neutrophil': np.int64(8891)}


In [2]:

import pandas as pd, hashlib, json
SEEDS=[42,1,7]
MODELS=[('convnext_tiny',8),('resnet50',8),('retfound_fullft',2)]
DATASETS=['PBC','Raabin_Train_TestA']
IDX_BNE=FULL.index('BNE')
IDX_SNE=FULL.index('SNE')
SUMMARY_FILE=O/'external_seed_results.csv'
OLD_SUMMARY=pd.read_csv(SUMMARY_FILE) if SUMMARY_FILE.exists() else None


def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''): h.update(b)
 return h.hexdigest()


def find_old_metric(dn,mn,seed):
 if OLD_SUMMARY is None: return None
 s=OLD_SUMMARY
 match=s[(s['dataset'].astype(str)==dn)&(s['model'].astype(str)==mn)&(s['seed'].astype(int)==seed)]
 if len(match)!=1: return None
 return float(match.iloc[0]['macro_f1'])


print('Eski tekil dis-test kayitlari:')
for dn in DATASETS:
 for m,bs in MODELS:
  for seed in SEEDS:
   old=O/dn/m/f'seed{seed}'/'probabilities.npy'
   print(dn,m,seed,'5-way probabilities:',old.exists())


Eski tekil dis-test kayitlari:
PBC convnext_tiny 42 5-way probabilities: True
PBC convnext_tiny 1 5-way probabilities: True
PBC convnext_tiny 7 5-way probabilities: True
PBC resnet50 42 5-way probabilities: True
PBC resnet50 1 5-way probabilities: True
PBC resnet50 7 5-way probabilities: True
PBC retfound_fullft 42 5-way probabilities: True
PBC retfound_fullft 1 5-way probabilities: True
PBC retfound_fullft 7 5-way probabilities: True
Raabin_Train_TestA convnext_tiny 42 5-way probabilities: True
Raabin_Train_TestA convnext_tiny 1 5-way probabilities: True
Raabin_Train_TestA convnext_tiny 7 5-way probabilities: True
Raabin_Train_TestA resnet50 42 5-way probabilities: True
Raabin_Train_TestA resnet50 1 5-way probabilities: True
Raabin_Train_TestA resnet50 7 5-way probabilities: True
Raabin_Train_TestA retfound_fullft 42 5-way probabilities: True
Raabin_Train_TestA retfound_fullft 1 5-way probabilities: False
Raabin_Train_TestA retfound_fullft 7 5-way probabilities: True


## 3. Aynı checkpoint ile 13 sınıfı yeniden hesaplayan fonksiyonlar

In [3]:

from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

def evaluate_13way(m, ds, bs=2):
    """Eski pre-processing ve siralamayla 13 sinif softmax olasiliklarini dondurur."""
    y_all=[]; five_all=[]; prob13_all=[]; paths=[]
    loader=DataLoader(ds,batch_size=bs,shuffle=False,num_workers=0,pin_memory=(DEV=='cuda'))
    with torch.inference_mode():
        for j,(x,y,path) in enumerate(loader,1):
            logits=m(x.to(DEV))
            if logits.ndim!=2 or logits.shape[1]!=13:
                raise RuntimeError(f'13 sinif bekleniyor, bulunan {tuple(logits.shape)}')
            # Eski Aşama 15C yöntemini TAM olarak tekrarla: softmax yalnızca bes logit üzerinde.
            q5=torch.softmax(logits[:,IDX],dim=1).float().cpu().numpy()
            q13=torch.softmax(logits,dim=1).float().cpu().numpy()
            y_all.append(y.numpy()); five_all.append(q5);prob13_all.append(q13); paths+=list(path)
            if j%100==0: print(f'    {j}/{len(loader)} batch')
    return (np.concatenate(y_all),np.concatenate(five_all),
            np.concatenate(prob13_all),paths)


def predicted_new(q13):
    # Bes yollu birlestirilmis skorlar; p(NEU)=p(BNE)+p(SNE).
    q=np.stack([q13[:,FULL.index(v)] for v in ['BA','EO','LY','MO']],axis=1)
    neu=(q13[:,IDX_BNE]+q13[:,IDX_SNE])[:,None]
    q=np.concatenate([q,neu],axis=1)
    q=q/q.sum(axis=1,keepdims=True)
    return q.argmax(axis=1),q


def class_f1(y,p):
    return f1_score(y,p,labels=np.arange(5),average=None,zero_division=0)


def validate_baseline(dn,mn,seed,y,q5):
    """Gecmis 5 sinifli tahminleri dogrular; uyusmazlik durumunda durur."""
    old_dir=O/dn/mn/f'seed{seed}'
    op=old_dir/'probabilities.npy'; oy=old_dir/'true.npy'; pred=old_dir/'pred.npy'
    old_f1=find_old_metric(dn,mn,seed)
    new_pred=q5.argmax(axis=1)
    score=float(f1_score(y,new_pred,labels=np.arange(5),average='macro',zero_division=0))
    if op.is_file() and oy.is_file():
        q_old=np.load(op,allow_pickle=False);y_old=np.load(oy,allow_pickle=False)
        if q_old.shape!=q5.shape or not np.array_equal(y_old,y):
            raise ValueError(f'{dn}/{mn}/{seed}: eski 5-class dosyalari ile satirlar uyusmuyor; analiz durduruldu')
        if not np.allclose(q_old,q5,atol=2e-5,rtol=2e-5):
            maxdiff=np.max(np.abs(q_old-q5))
            raise ValueError(f'{dn}/{mn}/{seed}: q5 != onceki probabilities.npy (maxdiff={maxdiff}); farkli checkpoint olabilir')
        if pred.is_file() and not np.array_equal(np.load(pred,allow_pickle=False),new_pred):
            raise ValueError(f'{dn}/{mn}/{seed}: Onceki pred.npy ile esit degil')
        return 'verified_against_saved_predictions',score
    if old_f1 is not None and abs(score-old_f1)<5e-5:
        return 'verified_against_seed_summary_only',score
    raise RuntimeError(
        f'{dn}/{mn}/seed{seed}: eski 5-class kaydi bulunamadi veya skor eslesmedi; '
        f'yeni macroF1={score:.8f}, eski summary={old_f1}. '
        'Bu seed icin uygun checkpointi elle dogrulamadan analizi raporlamayin.'
    )


## 4. PBC + Raabin: BNE+SNE analizi
18 koşunun tamamı için eski beş sınıflı olasılıklarla doğrulama yapılır; başarılı olanlar kaydedilir. İlerleme ara çıktılarla raporlanır. Grafikler / ham görüntüler dışarı aktarılmaz.

In [4]:

# No model training: fixed checkpoints ile ONLY forward inference.
import time
rows=[]; issues=[]
for dn in DATASETS:
    ds=D[dn]
    expected=10298 if dn=='PBC' else 14514
    if len(ds)!=expected:
        raise RuntimeError(f'{dn}: Goruntu sayisi {len(ds)}; onceki deney {expected}. Veri ayni degil.')
    for mn,bs in MODELS:
        for seed in SEEDS:
            print(f'\n==== {dn} | {mn} | seed{seed} ====')
            checkpoint=None
            try:
                checkpoint=ckpt(mn,seed)
                network=model(mn,checkpoint)
                t0=time.time()
                y,q5,q13,paths=evaluate_13way(network,ds,bs)
                del network
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                status,old_macro=validate_baseline(dn,mn,seed,y,q5)
                orig=q5.argmax(axis=1)
                alt,alt_prob=predicted_new(q13)
                f1base=class_f1(y,orig)
                f1new=class_f1(y,alt)
                if not np.array_equal(np.array([p for p,_ in ds.z],dtype=str),np.array(paths,dtype=str)):
                    raise RuntimeError('Dis test dosya sirasi degisti; islem durduruldu')
                output_dir=O_NEW/dn/mn/f'seed{seed}'
                output_dir.mkdir(parents=True, exist_ok=True)
                np.save(output_dir/'probabilities_13class.npy',q13)
                np.save(output_dir/'true_5class.npy',y)
                np.save(output_dir/'pred_baseline_5class.npy',orig)
                np.save(output_dir/'pred_BNEplusSNE_5class.npy',alt)
                with open(output_dir/'paths.txt','w',encoding='utf-8') as f:
                    f.write('\n'.join(paths))
                row=dict(dataset=dn,model=mn,seed=seed,n=len(y),
                    baseline_macro_f1=old_macro,
                    bne_sne_macro_f1=float(f1_score(y,alt,labels=np.arange(5),average='macro',zero_division=0)),
                    delta_macro_f1=float(f1_score(y,alt,labels=np.arange(5),average='macro',zero_division=0)-old_macro),
                    changed_predictions=int((orig!=alt).sum()),
                    baseline_neutrophil_recall=float((orig[y==4]==4).mean()),
                    bne_sne_neutrophil_recall=float((alt[y==4]==4).mean()),
                    verification=status,checkpoint=str(checkpoint),checkpoint_sha256=sha256(checkpoint),
                    seconds=round(time.time()-t0,1))
                for i,c in enumerate(NAMES):
                    row['baseline_f1_'+c]=float(f1base[i]);row['new_f1_'+c]=float(f1new[i])
                rows.append(row)
                print('OK',status,'base',f'{row["baseline_macro_f1"]:.6f}','BNE+SNE',f'{row["bne_sne_macro_f1"]:.6f}',
                      'delta',f'{row["delta_macro_f1"]:+.6f}','changed',row['changed_predictions'])
            except Exception as exc:
                issues.append({'dataset':dn,'model':mn,'seed':seed,'checkpoint':str(checkpoint),'error':str(exc)})
                print('ERROR, NOT REPORTED:',str(exc))
            finally:
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            pd.DataFrame(rows).to_csv(O_NEW/'external_BNE_SNE_per_seed.csv',index=False,encoding='utf-8-sig')
            pd.DataFrame(issues).to_csv(O_NEW/'issues.csv',index=False,encoding='utf-8-sig')

print('Tamamlanan dogrulanmis deney:',len(rows),'/18')
print('Hata sayisi:',len(issues))
if issues: display(pd.DataFrame(issues))



==== PBC | convnext_tiny | seed42 ====
resolved: convnext_tiny seed 42 -> C:\Users\sarah\Desktop\scı5-wbc\outputs\final\convnext_tiny\seed42\convnext_tiny_seed42_best.pt
checkpoint convnext_tiny_seed42_best.pt missing 0 unexpected 0
    100/1288 batch
    200/1288 batch
    300/1288 batch
    400/1288 batch
    500/1288 batch
    600/1288 batch
    700/1288 batch
    800/1288 batch
    900/1288 batch
    1000/1288 batch
    1100/1288 batch
    1200/1288 batch
ERROR, NOT REPORTED: PBC/convnext_tiny/42: q5 != onceki probabilities.npy (maxdiff=4.5180320739746094e-05); farkli checkpoint olabilir

==== PBC | convnext_tiny | seed1 ====
resolved: convnext_tiny seed 1 -> C:\Users\sarah\Desktop\scı5-wbc\outputs\final\convnext_tiny\seed1\convnext_tiny_seed1_best.pt
checkpoint convnext_tiny_seed1_best.pt missing 0 unexpected 0
    100/1288 batch
    200/1288 batch
    300/1288 batch
    400/1288 batch
    500/1288 batch
    600/1288 batch
    700/1288 batch
    800/1288 batch
    900/1288 batch


,dataset,model,seed,checkpoint,error
0,PBC,convnext_tiny,42,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,PBC/convnext_tiny/42: q5 != onceki probabiliti...
1,PBC,convnext_tiny,1,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,PBC/convnext_tiny/1: q5 != onceki probabilitie...
2,PBC,resnet50,42,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,PBC/resnet50/42: q5 != onceki probabilities.np...
3,PBC,resnet50,1,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,PBC/resnet50/1: q5 != onceki probabilities.npy...
4,PBC,resnet50,7,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,PBC/resnet50/7: q5 != onceki probabilities.npy...
5,PBC,retfound_fullft,42,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,Weights only load failed. This file can still ...
6,PBC,retfound_fullft,1,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,Weights only load failed. This file can still ...
7,PBC,retfound_fullft,7,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,Weights only load failed. This file can still ...
8,Raabin_Train_TestA,convnext_tiny,42,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,Raabin_Train_TestA/convnext_tiny/42: q5 != onc...
9,Raabin_Train_TestA,convnext_tiny,1,C:\Users\sarah\Desktop\scı5-wbc\outputs\final\...,Raabin_Train_TestA/convnext_tiny/1: q5 != once...


In [ ]:

if not rows:
    raise RuntimeError('Dogrulanmis hicbir dis test yok. issues.csv dosyasini inceleyin.')
per=pd.DataFrame(rows)
mean=(per.groupby(['dataset','model'],as_index=False)
      .agg(n_seeds=('seed','nunique'),
           baseline_macro_f1=('baseline_macro_f1','mean'),
           bne_sne_macro_f1=('bne_sne_macro_f1','mean'),
           delta_macro_f1=('delta_macro_f1','mean'),
           delta_sd=('delta_macro_f1','std'),
           mean_changed_predictions=('changed_predictions','mean')))
mean.to_csv(O_NEW/'external_BNE_SNE_summary.csv',index=False,encoding='utf-8-sig')
display(mean)
print('Kaydedildi:',O_NEW)
if len(rows)<18:
    print('DIKKAT: 18/18 deney dogrulanmadi. Makalede eksik seedleri tamamlanmis gibi raporlamayin.')
else:
    print('Tum 18 dataset/model/seed kaydi eski deneylerle eslesti.')
